# 01 - Fine-tune the extractor (FinSight, Rung 3)

Fine-tunes `deepset/deberta-v3-base-squad2` on the weakly labelled SQuAD 2.0 data (ADR-038, v2 labels
ADR-041). **Runs on Kaggle** (GPU T4, Internet on) with the private dataset `finsight-weaklabel`
attached. Nothing here is trained on the laptop.

1. **Smoke run** (`SLICE = 200`, `SEEDS = [13]`, `EPOCHS = 1`): passes when it runs end to end and
   writes `metrics.json` and weights. It says nothing about quality: 200 rows are about nine
   optimiser steps, too few to read a loss trend.
2. **Baseline** (`ZERO_SHOT = True`): no training; scores the model as downloaded on the dev split.
   This is the number a fine-tuned seed has to beat.
3. **Full run** (`SLICE = None`, one seed per notebook, `EPOCHS = 3`): the loss is logged as the
   average of the last 25 steps, the dev split is scored after every epoch (EM, token F1), and
   `final/` holds the weights of the **best** epoch by dev F1. If the session dies, run all cells
   again: it resumes from the last epoch checkpoint (`RESUME = True`).
4. NVM needs the FinSight normalizer, which is not on Kaggle: every epoch's dev predictions are
   saved to `predictions-<seed>.json` and NVM is computed on the laptop from them.

All of it is driven from the laptop: `uv run python -m finsight.weaklabel.kaggle push
smoke|baseline|13|42|2026`, then `status <run>` and `fetch <run>` (ADR-042, ADR-043). Weights go to
`models/extractor/seed-<n>/` and are never committed.


In [ ]:
# ---- parameters (the only cell you normally edit) -------------------------------------------
MODEL = "deepset/deberta-v3-base-squad2"   # starts from SQuAD 2.0, so it already knows "no answer"
DATASET = "finsight-weaklabel"            # name of the attached Kaggle dataset
SEEDS = [13, 42, 2026]                    # three seeds -> mean +- std in the report
SLICE = None                              # 200 = smoke test on train_200.jsonl / dev_200.jsonl
EPOCHS = 3
ZERO_SHOT = False                         # True = no training: score MODEL as downloaded (the baseline)
LOG_STEPS = 25                            # the logged loss is the average over these steps
LEARNING_RATE = 2e-5
BATCH_SIZE = 8                            # per device; lower to 4 if you see out-of-memory
GRAD_ACCUM = 2
MAX_LEN, STRIDE = 384, 128
MAX_ANSWER_TOKENS = 128                   # a list of four lead managers is longer than 30 tokens
FP16 = "auto"                             # "auto" = fp16 on GPU, fp32 on CPU; falls back to fp32 on NaN
RESUME = True                             # continue from the last checkpoint if one exists
OUT_ROOT = "/kaggle/working/extractor"    # weights and metrics are written here

In [ ]:
import json, os, random, re, shutil, string, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainerCallback,
                          TrainingArguments, default_data_collator, set_seed)

warnings.filterwarnings("ignore", category=UserWarning)
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob(f"**/{DATASET}"), None) or next(Path("/kaggle/input").iterdir())
else:  # local debugging: data/processed/kaggle/finsight-weaklabel
    DATA_DIR = Path("data/processed/kaggle") / DATASET
    OUT_ROOT = "extractor_out"
suffix = f"_{SLICE}" if SLICE else ""
TRAIN_FILE, DEV_FILE = Path(DATA_DIR) / f"train{suffix}.jsonl", Path(DATA_DIR) / f"dev{suffix}.jsonl"
print("data:", TRAIN_FILE, DEV_FILE, "| cuda:", torch.cuda.is_available())

In [ ]:
# ---- data -----------------------------------------------------------------------------------
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line]

train_rows, dev_rows = read_jsonl(TRAIN_FILE), read_jsonl(DEV_FILE)
assert not {r["ipo_id"] for r in train_rows} & {r["ipo_id"] for r in dev_rows}, "IPO leaked train->dev"
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev ids must be unique"
print(len(train_rows), "train rows from", len({r["ipo_id"] for r in train_rows}), "IPOs;",
      len(dev_rows), "dev rows from", len({r["ipo_id"] for r in dev_rows}), "IPOs")
print("answerable share (train):", round(sum(bool(r["answers"]["text"]) for r in train_rows) / len(train_rows), 2))

In [ ]:
# ---- tokenising (offsets map answer characters to tokens) -----------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL)
PAD_RIGHT = tokenizer.padding_side == "right"


def prepare_train_features(rows):
    """One feature per window of at most MAX_LEN tokens; windows without the answer point at token 0."""
    enc = tokenizer([r["question"].lstrip() for r in rows], [r["context"] for r in rows],
                    truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    owner = enc.pop("overflow_to_sample_mapping")
    offsets = enc.pop("offset_mapping")
    enc["start_positions"], enc["end_positions"] = [], []
    for i, offs in enumerate(offsets):
        ids = enc["input_ids"][i]
        cls = ids.index(tokenizer.cls_token_id)
        seq = enc.sequence_ids(i)
        answers = rows[owner[i]]["answers"]
        if not answers["text"]:
            enc["start_positions"].append(cls); enc["end_positions"].append(cls); continue
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        ts = 0
        while seq[ts] != 1: ts += 1
        te = len(ids) - 1
        while seq[te] != 1: te -= 1
        if not (offs[ts][0] <= start_char and offs[te][1] >= end_char):
            enc["start_positions"].append(cls); enc["end_positions"].append(cls); continue
        while ts < len(offs) and offs[ts][0] <= start_char: ts += 1
        enc["start_positions"].append(ts - 1)
        while offs[te][1] >= end_char: te -= 1
        enc["end_positions"].append(te + 1)
    return enc


def prepare_eval_features(rows):
    enc = tokenizer([r["question"].lstrip() for r in rows], [r["context"] for r in rows],
                    truncation="only_second", max_length=MAX_LEN, stride=STRIDE,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    owner = enc.pop("overflow_to_sample_mapping")
    enc["example_idx"] = list(owner)
    for i in range(len(enc["input_ids"])):
        seq = enc.sequence_ids(i)
        enc["offset_mapping"][i] = [o if seq[k] == 1 else None for k, o in enumerate(enc["offset_mapping"][i])]
    return enc


class Features(Dataset):
    def __init__(self, enc, keys):
        self.enc, self.keys = enc, keys
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        return {k: torch.tensor(self.enc[k][i]) for k in self.keys}

In [ ]:
# ---- metrics (mirrors finsight/evaluate/metrics.py: SQuAD EM and token F1) ------------------
_ART = re.compile(r"\b(a|an|the)\b")
def normalize_text(t):
    t = "".join(c for c in t.lower() if c not in set(string.punctuation))
    return " ".join(_ART.sub(" ", t).split())
def exact_match(p, g):
    return normalize_text(p) == normalize_text(g)
def token_f1(p, g):
    p, g = normalize_text(p).split(), normalize_text(g).split()
    if not p and not g: return 1.0
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)


def postprocess(rows, feats, start_logits, end_logits):
    """Best span per example over all its windows: {"text", "start"}; "" and -1 = no answer."""
    by_example = {}
    for fi, ei in enumerate(feats["example_idx"]):
        by_example.setdefault(ei, []).append(fi)
    out = []
    for ei, row in enumerate(rows):
        best_score, best_text, best_start, null_score = -1e9, "", -1, 1e9
        for fi in by_example.get(ei, []):
            s, e, offs = start_logits[fi], end_logits[fi], feats["offset_mapping"][fi]
            null_score = min(null_score, s[0] + e[0])
            for si in np.argsort(s)[-20:][::-1]:
                for ej in np.argsort(e)[-20:][::-1]:
                    if offs[si] is None or offs[ej] is None or ej < si or ej - si + 1 > MAX_ANSWER_TOKENS:
                        continue
                    score = s[si] + e[ej]
                    if score > best_score:
                        best_score, best_start = score, int(offs[si][0])
                        best_text = row["context"][offs[si][0]:offs[ej][1]]
        answered = null_score < best_score
        out.append({"text": best_text if answered else "", "start": best_start if answered else -1})
    return out


def squad_scores(rows, predictions):
    has = [r for r, _ in zip(rows, predictions) if r["answers"]["text"]]
    pairs = list(zip(rows, predictions))
    pos = [(r["answers"]["text"][0], p) for r, p in pairs if r["answers"]["text"]]
    neg = [p for r, p in pairs if not r["answers"]["text"]]
    return {
        "n_dev": len(rows), "n_answerable": len(pos), "n_unanswerable": len(neg),
        "HasAns_EM": float(np.mean([exact_match(p, g) for g, p in pos])) if pos else None,
        "HasAns_F1": float(np.mean([token_f1(p, g) for g, p in pos])) if pos else None,
        "NoAns_acc": float(np.mean([p == "" for p in neg])) if neg else None,
        "EM": float(np.mean([exact_match(p, r["answers"]["text"][0] if r["answers"]["text"] else "")
                             for r, p in pairs])),
        "F1": float(np.mean([token_f1(p, r["answers"]["text"][0] if r["answers"]["text"] else "")
                             for r, p in pairs])),
    }

In [ ]:
# ---- dev evaluation (used after every epoch and for the zero-shot baseline) ------------------
EVAL_ENC = prepare_eval_features(dev_rows)
EVAL_KEYS = ["input_ids", "attention_mask"] + (["token_type_ids"] if "token_type_ids" in EVAL_ENC else [])


def predict_dev(model):
    """Dev answers from the model as it is now, in fp32: one {"text", "start"} per dev row."""
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    starts, ends = [], []
    with torch.no_grad():
        for lo in range(0, len(EVAL_ENC["input_ids"]), 32):
            batch = {k: torch.tensor(EVAL_ENC[k][lo:lo + 32], device=device) for k in EVAL_KEYS}
            with torch.autocast(device.type, enabled=False):
                out = model(**batch)
            starts.append(out.start_logits.float().cpu().numpy())
            ends.append(out.end_logits.float().cpu().numpy())
    if was_training:
        model.train()
    return postprocess(dev_rows, EVAL_ENC, np.concatenate(starts), np.concatenate(ends))


def score_dev(model):
    preds = predict_dev(model)
    return squad_scores(dev_rows, [p["text"] for p in preds]), {r["id"]: p for r, p in zip(dev_rows, preds)}


class DevEval(TrainerCallback):
    """After each epoch: score dev, remember the predictions, keep the best epoch's weights in final/."""

    def __init__(self, out_dir):
        self.out_dir, self.file = out_dir, out_dir / "epochs.json"
        self.log = {"epochs": [], "predictions": {}, "best_epoch": None, "best_f1": -1.0}
        if RESUME and self.file.exists():
            self.log = json.loads(self.file.read_text(encoding="utf-8"))

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        scores, preds = score_dev(model)
        self.log["epochs"] = [e for e in self.log["epochs"] if e["epoch"] != epoch] + [{"epoch": epoch, **scores}]
        self.log["predictions"][str(epoch)] = preds
        if scores["F1"] > self.log["best_f1"]:
            self.log["best_f1"], self.log["best_epoch"] = scores["F1"], epoch
            model.save_pretrained(str(self.out_dir / "final"))
            tokenizer.save_pretrained(str(self.out_dir / "final"))
        self.file.write_text(json.dumps(self.log), encoding="utf-8")
        print(f"epoch {epoch}: dev EM {scores['EM']:.4f} F1 {scores['F1']:.4f} "
              f"HasAns_F1 {scores['HasAns_F1']:.4f} NoAns_acc {scores['NoAns_acc']:.4f} "
              f"(best so far: epoch {self.log['best_epoch']})")


# ---- one training run (a seed), with epoch checkpoints, resume and fp32 fallback -------------
def run_seed(seed, fp16):
    set_seed(seed)
    out_dir = Path(OUT_ROOT) / f"seed-{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    model = AutoModelForQuestionAnswering.from_pretrained(MODEL)
    train_enc = prepare_train_features(train_rows)
    train_ds = Features(train_enc, ["input_ids", "attention_mask", "token_type_ids", "start_positions", "end_positions"]
                        if "token_type_ids" in train_enc else ["input_ids", "attention_mask", "start_positions", "end_positions"])
    args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"), num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        weight_decay=0.01, warmup_ratio=0.1, seed=seed, fp16=fp16, save_strategy="epoch",
        save_total_limit=1, logging_steps=1 if SLICE else LOG_STEPS, report_to="none", dataloader_num_workers=2,
    )
    dev_eval = DevEval(out_dir)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=default_data_collator,
                      callbacks=[dev_eval])
    checkpoints = sorted((out_dir / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    resume = str(checkpoints[-1]) if (RESUME and checkpoints) else None
    if resume:
        print("resuming from", resume)
    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    history = [{"step": h["step"], "epoch": round(h["epoch"], 3), "loss": h["loss"]}
               for h in trainer.state.log_history if "loss" in h]
    if history and not all(np.isfinite(h["loss"]) for h in history):
        raise FloatingPointError("loss is NaN")
    runtime = time.time() - t0

    log = dev_eval.log
    best = next(e for e in log["epochs"] if e["epoch"] == log["best_epoch"])
    metrics = {k: v for k, v in best.items() if k != "epoch"} | {
        "seed": seed, "model": MODEL, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE * GRAD_ACCUM, "max_len": MAX_LEN, "stride": STRIDE, "fp16": bool(fp16),
        "max_answer_tokens": MAX_ANSWER_TOKENS, "n_gpu": torch.cuda.device_count(),
        "n_train": len(train_rows), "train_runtime_s": round(runtime),
        "final_train_loss": history[-1]["loss"] if history else None,
        "slice": SLICE, "log_steps": 1 if SLICE else LOG_STEPS,
        "best_epoch": log["best_epoch"], "per_epoch": sorted(log["epochs"], key=lambda e: e["epoch"]),
        "loss_history": history,
    }
    (Path(OUT_ROOT) / f"metrics-{seed}.json").write_text(json.dumps(metrics, indent=1), encoding="utf-8")
    (Path(OUT_ROOT) / f"predictions-{seed}.json").write_text(json.dumps(log["predictions"]), encoding="utf-8")
    shutil.rmtree(out_dir / "checkpoints", ignore_errors=True)  # the output keeps the best weights only
    del trainer, model
    torch.cuda.empty_cache()
    return metrics


In [ ]:
# ---- zero-shot baseline, or every seed -------------------------------------------------------
use_fp16 = torch.cuda.is_available() if FP16 == "auto" else bool(FP16)
all_metrics = []
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
if ZERO_SHOT:
    base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL)
    base_model = base_model.to("cuda" if torch.cuda.is_available() else "cpu")
    scores, preds = score_dev(base_model)
    baseline = scores | {"model": MODEL, "zero_shot": True, "slice": SLICE, "max_len": MAX_LEN,
                         "stride": STRIDE, "max_answer_tokens": MAX_ANSWER_TOKENS}
    (Path(OUT_ROOT) / "metrics-baseline.json").write_text(json.dumps(baseline, indent=1), encoding="utf-8")
    (Path(OUT_ROOT) / "predictions-baseline.json").write_text(json.dumps({"0": preds}), encoding="utf-8")
    print("BASELINE OK:", {k: baseline[k] for k in ("EM", "F1", "HasAns_F1", "NoAns_acc")})
for seed in ([] if ZERO_SHOT else SEEDS):
    done = Path(OUT_ROOT) / f"metrics-{seed}.json"
    if RESUME and done.exists():
        print(f"seed {seed}: already finished, skipping")
        all_metrics.append(json.loads(done.read_text(encoding="utf-8")))
        continue
    try:
        m = run_seed(seed, use_fp16)
    except (FloatingPointError, RuntimeError) as exc:
        if not use_fp16:
            raise
        print(f"seed {seed}: fp16 failed ({exc}); retrying in fp32 from scratch")
        shutil.rmtree(Path(OUT_ROOT) / f"seed-{seed}", ignore_errors=True)
        m = run_seed(seed, False)
    print({k: m[k] for k in ("seed", "best_epoch", "EM", "F1", "HasAns_F1", "NoAns_acc")})
    all_metrics.append(m)


In [ ]:
# ---- summary and smoke check ----------------------------------------------------------------
if all_metrics:
    summary = {
        "seeds": [m["seed"] for m in all_metrics],
        **{name: {"mean": float(np.mean([m[name] for m in all_metrics])),
                  "std": float(np.std([m[name] for m in all_metrics]))}
           for name in ("EM", "F1", "HasAns_F1", "NoAns_acc") if all(m[name] is not None for m in all_metrics)},
    }
    (Path(OUT_ROOT) / "metrics.json").write_text(json.dumps({"summary": summary, "runs": all_metrics}, indent=1), encoding="utf-8")
    print(json.dumps(summary, indent=1))
if SLICE and not ZERO_SHOT:
    assert all(np.isfinite(m["F1"]) for m in all_metrics)
    assert all((Path(OUT_ROOT) / f"seed-{m['seed']}" / "final" / "config.json").exists() for m in all_metrics)
    print("SMOKE OK: the notebook ran end to end and wrote metrics and weights.")
